# 5. Run the Bayesian model in your browser

This notebook fits the pipeline's **joint two-period Beta-Binomial model** to the
synthetic H3 data, live, in your browser. It is the same model as in
`pipeline/analyzers/bayesian.py` (`run_two_period_model`).

**Be patient.** The browser cannot use the fast compiled back-ends that PyMC normally
relies on, so sampling runs in pure Python. With the reduced settings below it took
about **6 minutes** in our test (slower computers and browsers take longer); the
production settings (4 chains × 1,500 draws) would take about an hour. On a normal Python installation the same code runs in under a
minute.

In [1]:
import os, sys
if sys.platform == "emscripten":
    # No numba / C compiler in the browser: run PyTensor in pure-Python mode.
    os.environ["PYTENSOR_FLAGS"] = "mode=FAST_COMPILE,linker=py,cxx="
    import piplite
    await piplite.install(["numpy", "scipy", "pandas", "geopandas", "matplotlib", "xarray",
                           "filelock", "setuptools", "cachetools", "cloudpickle", "rich",
                           "threadpoolctl", "typing-extensions", "arviz"])
    # pytensor and pymc pull in numba, which does not exist for the browser.
    await piplite.install("pytensor", deps=False)
    await piplite.install("pymc", deps=False)

In [2]:
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
import arviz as az
from hotspots_lite import build_info, load_results

CHAINS, DRAWS, TUNE = 2, 400, 400     # production: 4 chains, 1,500 draws, 1,500 tuning steps

info = build_info()
cells = load_results()
d = cells[cells["tests_curr"].fillna(0) > 0].reset_index(drop=True)
y, n = d["recent_curr"].astype(int).to_numpy(), d["tests_curr"].astype(int).to_numpy()
y_h = d["recent_hist"].fillna(0).astype(int).to_numpy()
n_h = d["tests_hist"].fillna(0).astype(int).to_numpy()
has_hist = n_h > 0
K = len(d)

national = y.sum() / n.sum()
prior_mu = float(np.log(np.clip(national, 1e-4, 0.5) / (1 - np.clip(national, 1e-4, 0.5))))

# Testing-intensity normalisation (standardised log test-months, pooled over both windows).
li_c = np.log1p(d["test_months_curr"].fillna(0).to_numpy(float))
li_h = np.log1p(d["test_months_hist"].fillna(0).to_numpy(float))
pool = np.concatenate([li_c, li_h])
z_c, z_h = (li_c - pool.mean()) / pool.std(), (li_h - pool.mean()) / pool.std()
print(f"{K} cells, national recent share {national:.3%}; {has_hist.sum()} cells have a baseline")

26 cells, national recent share 1.481%; 26 cells have a baseline


## The model

For area *i* and window *t* (baseline or current):

logit *p*<sub>it</sub> = μ<sub>t</sub> + *a*<sub>i</sub> + δ<sub>i</sub>·[t = current] + β·*z*<sub>it</sub>

* μ<sub>t</sub>: national level in each window;
* *a*<sub>i</sub>: the area's own level, pooled across areas;
* δ<sub>i</sub>: the area's change between windows, pooled towards a common change;
* β·*z*<sub>it</sub>: adjustment for testing effort (test-months), so the trend is not
  driven by sites opening or closing;
* recent counts are Beta-Binomial with a concentration κ learned from the data.

In [3]:
with pm.Model() as model:
    mu_hist = pm.Normal("mu_hist", mu=prior_mu, sigma=1.5)
    mu_curr = pm.Normal("mu_curr", mu=prior_mu, sigma=1.5)
    sigma_a = pm.HalfNormal("sigma_a", sigma=2)
    mu_delta = pm.Normal("mu_delta", mu=0.0, sigma=1.0)
    sigma_delta = pm.HalfNormal("sigma_delta", sigma=1.0)
    a = pm.Deterministic("a", sigma_a * pm.Normal("a_offset", 0, 1, shape=K))
    delta = pm.Deterministic("delta", mu_delta + sigma_delta * pm.Normal("d_offset", 0, 1, shape=K))
    beta_intensity = pm.Normal("beta_intensity", mu=0.0, sigma=1.0)
    p_hist = pm.Deterministic("p_hist", pm.math.invlogit(mu_hist + a + beta_intensity * z_h))
    p_curr = pm.Deterministic("p_curr", pm.math.invlogit(mu_curr + a + delta + beta_intensity * z_c))
    kappa = pm.Gamma("kappa", alpha=2, beta=0.01)
    pm.BetaBinomial("y_obs", alpha=p_curr * kappa, beta=(1 - p_curr) * kappa, n=n, observed=y)
    bb_h = pm.BetaBinomial.dist(alpha=p_hist * kappa, beta=(1 - p_hist) * kappa, n=np.maximum(n_h, 1))
    pm.Potential("y_hist_ll", (pm.logp(bb_h, y_h) * has_hist.astype(float)).sum())

    idata = pm.sample(draws=DRAWS, tune=TUNE, chains=CHAINS, cores=1, target_accept=0.95,
                      random_seed=42, progressbar=False)

s = az.summary(idata, var_names=["mu_curr", "sigma_a", "sigma_delta", "kappa"])
print(f"max R-hat {s['r_hat'].max():.3f} (needs < 1.01), min bulk ESS {s['ess_bulk'].min():.0f} (needs > 400)")
s

Initializing NUTS using jitter+adapt_diag...


Sequential sampling (2 chains in 1 job)


NUTS: [mu_hist, mu_curr, sigma_a, mu_delta, sigma_delta, a_offset, d_offset, beta_intensity, kappa]


Sampling 2 chains for 400 tune and 400 draw iterations (800 + 800 draws total) took 7 seconds.


We recommend running at least 4 chains for robust computation of convergence diagnostics


The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


max R-hat 1.005 (needs < 1.01), min bulk ESS 190 (needs > 400)


,mean,sd,eti89_lb,eti89_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
mu_curr,-4.53,0.91,-5.9,-3,565,607,1.00,0.039,0.028
sigma_a,1.18,0.46,0.48,2,206,173,1.00,0.031,0.019
sigma_delta,0.78,0.51,0.069,1.7,189,218,1.01,0.034,0.018
kappa,2.08e+02,1.35e+02,63,470,542,564,1.00,5.5,5.8


**Check the diagnostics before reading anything else.** With the reduced browser
settings the sampler often stops short of the pipeline's thresholds (R-hat below 1.01,
effective sample size above 400). The pipeline treats such a run as unreliable and
does not write a report. Here we carry on for teaching purposes; for real use, raise
`CHAINS`, `DRAWS` and `TUNE` towards the production values (4, 1,500, 1,500) and run
on a normal Python installation.

## From the posterior to labels

The level compares each area's current share with the national share **without that
area**; the trend reads the change δ directly. Each axis uses the 0.80 rule, raised
when a stricter cutoff keeps the expected share of false calls within 5 %. The presence
and cold-cell gates come last. (The pipeline has one more gate that withholds a trend
when an area's history is too thin; it needs extra inputs and is left out here, so a
few labels can differ from the pipeline's.)

In [4]:
pc = idata.posterior["p_curr"].values.reshape(-1, K)
dl = idata.posterior["delta"].values.reshape(-1, K)
loo = (y.sum() - y) / (n.sum() - n)

exc = {
    "level_high": (pc > loo).mean(0), "level_low": (pc < loo).mean(0),
    "trend_up": (dl > 0).mean(0), "trend_down": (dl < 0).mean(0),
}

def fdr_cutoff(p, start=0.80, max_fdr=0.05, step=0.005, ceiling=0.99):
    """Lowest cutoff >= start at which the Bayesian FDR of the flagged areas is <= 5 %."""
    t = start
    while t <= ceiling:
        sel = p[p > t]
        if len(sel) == 0:
            break
        if np.mean(1 - sel) <= max_fdr:
            return t
        t += step
    return start

flags = {k: v > fdr_cutoff(v) for k, v in exc.items()}
level = np.where(flags["level_high"], "high", np.where(flags["level_low"], "low", "norm"))
trend = np.where(flags["trend_up"], "high", np.where(flags["trend_down"], "low", "norm"))
label_map = {("high", "high"): "Established hotspot", ("high", "norm"): "Emerging hotspot",
             ("high", "low"): "Normal", ("norm", "high"): "Stable high-burden",
             ("norm", "norm"): "Normal", ("norm", "low"): "Below national",
             ("low", "high"): "Declining from high-burden", ("low", "norm"): "Emerging decrease",
             ("low", "low"): "Below national"}
hot = {"Established hotspot", "Emerging hotspot", "Stable high-burden"}
labels = []
for i in range(K):
    if y[i] == 0 and y_h[i] == 0:
        labels.append("Normal")                       # cold-cell gate
        continue
    lab = label_map[(trend[i], level[i])]
    labels.append("Normal" if lab in hot and y[i] < 2 else lab)   # presence gate
d["live_label"] = labels
d["P(level high)"] = exc["level_high"].round(3)
d["P(trend up)"] = exc["trend_up"].round(3)
print(f"Agreement with the precomputed pipeline labels: {(d.live_label == d.label).mean():.0%}")
d[["name", "tests_curr", "recent_curr", "P(level high)", "P(trend up)", "live_label", "label"]].sort_values(
    "P(level high)", ascending=False).head(12)

Agreement with the precomputed pipeline labels: 92%


,name,tests_curr,recent_curr,P(level high),P(trend up),live_label,label
19,Hex in Drabivska,7.0,2.0,0.972,0.588,Stable high-burden,Elevated vs national (trend uncertain)
12,Hex in Kozeletska,41.0,4.0,0.944,0.670,Stable high-burden,Stable high-burden
6,Hex in Kostiantynivska,93.0,4.0,0.942,0.472,Stable high-burden,Stable high-burden
23,Hex in Verbkivska,138.0,7.0,0.930,0.619,Normal,Stable high-burden
7,Hex in Lozuvatska,105.0,3.0,0.660,0.539,Normal,Normal
21,Hex in Shakhivska,10.0,1.0,0.634,0.582,Normal,Normal
22,Hex in Boikivska,11.0,1.0,0.630,0.548,Normal,Normal
17,Hex in Talalaivska,4.0,0.0,0.411,0.406,Normal,Normal
15,Hex in Chutivska,3.0,0.0,0.386,0.422,Normal,Normal
4,Hex in Mykolaivska,6.0,0.0,0.356,0.419,Normal,Normal


Differences from the precomputed labels come from four places: fewer posterior draws
here, the omitted thin-history gate, ordinary Monte Carlo noise near the 0.80 cutoff,
and the false-discovery step. That last one can drop a borderline area even above
0.80: if the areas above 0.80 together carry more than 5 % expected false calls, the
cutoff rises until they do not, and the weakest of them falls out. An area whose label flips between runs is, by definition, a borderline case;
the reliability rating and the watch-list are there to show that.